# Collaborative Filtering — Step by Step

Build a **matrix factorization** collaborative filtering model on the Last.fm 360K dataset (user–artist play counts).

**Steps:**
1. Load and inspect the data
2. Prepare data: map users/items to indices, create train/test split
3. Implement matrix factorization (latent factors + SGD)
4. Train the model
5. Evaluate: RMSE and recall@k

## 1. Load the data

In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

DATASET_DIR = Path("../dataset")
PLAYS_FILE = DATASET_DIR / "usersha1-artmbid-artname-plays.tsv"

# Use a sample for fast iteration (full file is ~17.5M rows)
NROWS = 200_000

plays = pd.read_csv(
    PLAYS_FILE,
    sep="\t",
    nrows=NROWS,
    header=None,
    names=["user_sha1", "artist_mbid", "artist_name", "plays"],
)

print("Shape:", plays.shape)
print(plays.head(10))

Shape: (200000, 4)
                                  user_sha1  \
0  00000c289a1829a808ac09c00daf10bc3c4e223b   
1  00000c289a1829a808ac09c00daf10bc3c4e223b   
2  00000c289a1829a808ac09c00daf10bc3c4e223b   
3  00000c289a1829a808ac09c00daf10bc3c4e223b   
4  00000c289a1829a808ac09c00daf10bc3c4e223b   
5  00000c289a1829a808ac09c00daf10bc3c4e223b   
6  00000c289a1829a808ac09c00daf10bc3c4e223b   
7  00000c289a1829a808ac09c00daf10bc3c4e223b   
8  00000c289a1829a808ac09c00daf10bc3c4e223b   
9  00000c289a1829a808ac09c00daf10bc3c4e223b   

                            artist_mbid              artist_name  plays  
0  3bd73256-3905-4f3a-97e2-8b341527f805          betty blowtorch   2137  
1  f2fb0ff0-5679-42ec-a55c-15109ce6e320                die Ärzte   1099  
2  b3ae82c2-e60b-4551-a76d-6620f1b456aa        melissa etheridge    897  
3  3d6bbeb7-f90e-4d10-b440-e153c0d10b53                elvenking    717  
4  bbd2ffd7-17f4-4506-8572-c1ea58c3f9a8     juliette & the licks    706  
5  8bfac288-ccc5-44

## 2. Prepare data for collaborative filtering

- Map each `user_sha1` and `artist_mbid` to integer IDs (required for indexing).
- Optionally filter to users/artists with enough interactions to reduce noise.
- Use **rating** = play count (or log-scaled). We'll use \(\log(1 + \text{plays})\) to dampen extremes.

In [13]:
# Drop rows with missing artist id (some rows have empty mbid)
plays = plays.dropna(subset=["artist_mbid"])
plays = plays[plays["artist_mbid"].str.len() > 0]

# Rating: log(1 + plays) so high play counts don't dominate
plays["rating"] = np.log1p(plays["plays"].astype(float))

# Map to integer indices
user_ids = plays["user_sha1"].astype("category")
item_ids = plays["artist_mbid"].astype("category")
plays["user_idx"] = user_ids.cat.codes.values
plays["item_idx"] = item_ids.cat.codes.values

n_users = plays["user_idx"].max() + 1
n_items = plays["item_idx"].max() + 1

# Mapping from item_idx -> artist name (for demo recommendations)
idx_to_artist = plays.drop_duplicates("item_idx").set_index("item_idx")["artist_name"].to_dict()

print(f"Users: {n_users}, Items: {n_items}, Interactions: {len(plays)}")
print(plays[["user_idx", "item_idx", "plays", "rating"]].head(10))

Users: 4089, Items: 31717, Interactions: 197473
   user_idx  item_idx  plays    rating
0         0      7461   2137  7.667626
1         0     30098   1099  7.003065
2         0     22275    897  6.800170
3         0      7674    717  6.576470
4         0     23288    706  6.561031
5         0     17393    691  6.539586
6         0     12562    545  6.302619
7         0      4247    507  6.230481
8         0     24578    424  6.052089
9         0       785    403  6.001415


## 3. Train / test split

Randomly hold out 20% of interactions for testing. For each user we need at least one interaction in train (so we can learn their embedding).

In [6]:
np.random.seed(42)

# Random permutation and 80/20 split
idx = np.random.permutation(len(plays))
split = int(0.8 * len(plays))
train_idx, test_idx = idx[:split], idx[split:]

train_df = plays.iloc[train_idx].copy()
test_df = plays.iloc[test_idx].copy()

# Keep only test pairs whose user and item appear in train (cold-start skip)
train_users = set(train_df["user_idx"])
train_items = set(train_df["item_idx"])
test_df = test_df[
    test_df["user_idx"].isin(train_users) & test_df["item_idx"].isin(train_items)
].copy()

print(f"Train size: {len(train_df)}, Test size: {len(test_df)}")

Train size: 157978, Test size: 35714


## 4. Matrix factorization model

We learn:
- **P** (n_users × dim): user latent vectors
- **Q** (n_items × dim): item latent vectors

Prediction: \(\hat{r}_{ui} = P_u \cdot Q_i^T\)  
Train with **SGD** to minimize MSE: \( (r_{ui} - \hat{r}_{ui})^2 + \lambda (\|P_u\|^2 + \|Q_i\|^2) \)

In [7]:
def train_mf(
    train_df,
    n_users,
    n_items,
    dim=32,
    lr=0.01,
    reg=0.02,
    n_epochs=20,
    seed=42,
):
    np.random.seed(seed)
    P = np.random.randn(n_users, dim) * 0.1
    Q = np.random.randn(n_items, dim) * 0.1

    user_idx = train_df["user_idx"].values
    item_idx = train_df["item_idx"].values
    ratings = train_df["rating"].values.astype(np.float64)
    n = len(ratings)

    for epoch in range(n_epochs):
        perm = np.random.permutation(n)
        epoch_loss = 0.0
        for pos in perm:
            u, i, r = user_idx[pos], item_idx[pos], ratings[pos]
            pred = P[u] @ Q[i]
            err = r - pred
            epoch_loss += err * err
            P[u] += lr * (err * Q[i] - reg * P[u])
            Q[i] += lr * (err * P[u] - reg * Q[i])
        rmse = np.sqrt(epoch_loss / n)
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch {epoch + 1}/{n_epochs} — Train RMSE: {rmse:.4f}")

    return P, Q

## 5. Train the model

In [8]:
P, Q = train_mf(
    train_df,
    n_users=n_users,
    n_items=n_items,
    dim=32,
    lr=0.01,
    reg=0.02,
    n_epochs=20,
)

Epoch 1/20 — Train RMSE: 4.6866
Epoch 5/20 — Train RMSE: 2.0620
Epoch 10/20 — Train RMSE: 1.0039
Epoch 15/20 — Train RMSE: 0.6408
Epoch 20/20 — Train RMSE: 0.4883


## 6. Evaluate

- **RMSE** on the test set (prediction accuracy).
- **Recall@k**: for each test user, rank all items by predicted score; what fraction of their held-out items appear in top-k?

In [9]:
# Test RMSE
test_user = test_df["user_idx"].values
test_item = test_df["item_idx"].values
test_rating = test_df["rating"].values
preds = (P[test_user] * Q[test_item]).sum(axis=1)
test_rmse = np.sqrt(np.mean((test_rating - preds) ** 2))
print(f"Test RMSE: {test_rmse:.4f}")

Test RMSE: 1.0366


In [10]:
def recall_at_k(test_df, P, Q, k=10, n_sample_items=1000):
    """For each user in test, compute recall@k. Candidate set = true items + random sample."""
    test_by_user = test_df.groupby("user_idx")["item_idx"].apply(list).to_dict()
    recalls = []
    for u, true_items in test_by_user.items():
        if not true_items:
            continue
        true_set = set(true_items)
        # Candidates: true items + random sample of others so we can rank
        others = [i for i in range(n_items) if i not in true_set]
        n_others = min(len(others), n_sample_items)
        sample_others = np.random.choice(others, size=n_others, replace=False) if n_others > 0 else []
        candidates = np.array(list(true_set) + list(sample_others))
        if len(candidates) == 0:
            continue
        scores = P[u] @ Q[candidates].T
        top_k = candidates[np.argsort(-scores)[:k]]
        hits = len(set(top_k) & true_set)
        recalls.append(hits / min(len(true_items), k))
    return np.mean(recalls) if recalls else 0.0

np.random.seed(123)
print(f"Recall@10 (sampled): {recall_at_k(test_df, P, Q, k=10):.4f}")

Recall@10 (sampled): 0.1550


## Demo: Recommend artists for a user

Pick a user, exclude artists they've already played, and show **top-k recommended artists** by predicted score.

In [12]:
# Item index -> artist name (needed for display)
idx_to_artist = plays.drop_duplicates("item_idx").set_index("item_idx")["artist_name"].to_dict()


def recommend_for_user(user_idx, P, Q, train_df, idx_to_artist, k=10):
    """Top-k artist recommendations for a user (excluding artists they already played)."""
    seen = set(train_df[train_df["user_idx"] == user_idx]["item_idx"].values)
    scores = P[user_idx] @ Q.T  # score every item
    # Sort by score descending, skip already-seen
    ranked = np.argsort(-scores)
    recs = []
    for i in ranked:
        if i not in seen:
            recs.append((i, float(scores[i])))
            if len(recs) >= k:
                break
    return recs


# Demo: recommend for user 0 (or change to any user_idx in [0, n_users))
demo_user = 0
top_k = 10
recs = recommend_for_user(demo_user, P, Q, train_df, idx_to_artist, k=top_k)

# Show what they already listen to (top 5 by plays)
already = (
    train_df[train_df["user_idx"] == demo_user]
    .nlargest(5, "plays")[["artist_name", "plays"]]
    .values
)
print("User already listens to (top 5 by plays):")
for name, count in already:
    print(f"  • {name} ({int(count):,} plays)")
print()

print(f"Recommended artists (top {top_k}):")
for rank, (item_idx, score) in enumerate(recs, 1):
    name = idx_to_artist.get(item_idx, f"item_{item_idx}")
    print(f"  {rank:2}. {name}  (score: {score:.3f})")

User already listens to (top 5 by plays):
  • betty blowtorch (2,137 plays)
  • die Ärzte (1,099 plays)
  • melissa etheridge (897 plays)
  • elvenking (717 plays)
  • juliette & the licks (706 plays)

Recommended artists (top 10):
   1. they might be giants  (score: 6.808)
   2. arctic monkeys  (score: 6.791)
   3. Элизиум  (score: 6.752)
   4. typ:t.u.r.b.o.  (score: 6.729)
   5. misfits  (score: 6.612)
   6. sdp  (score: 6.576)
   7. alkaline trio  (score: 6.533)
   8. o teatro mágico  (score: 6.448)
   9. wise guys  (score: 6.422)
  10. nofx  (score: 6.391)


## 7. Next steps

- Increase `NROWS` or use full dataset for better accuracy.
- Tune `dim`, `lr`, `reg`, `n_epochs`.
- Try **BPR** (Bayesian Personalized Ranking) for ranking loss instead of MSE.
- Move on to **LightGCN** (graph convolution on user–item graph) for stronger performance.